# House Price Prediction using Artificial Neural Networks

This cleaned portfolio notebook uses the modular project code instead of duplicating
all logic. It reproduces the workflow for the California Housing dataset and keeps
the deployed model scope explicit: **median census block-group house value**, not
an individual-property appraisal.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    DATA_PATH, FEATURE_COLUMNS, METADATA_PATH, MODEL_PATH, OUTPUT_DIR,
    SCALER_PATH, TARGET_MULTIPLIER
)
from src.data_preprocessing import (
    build_iqr_outlier_report, load_housing_data, split_housing_data
)
from src.feature_engineering import add_optional_exploratory_features
from src.model_evaluation import regression_metrics
from src.prediction_pipeline import PredictionPipeline

pd.set_option("display.max_columns", None)


## 1. Load and inspect the actual project dataset

In [ ]:
data = load_housing_data(DATA_PATH)
print("Shape:", data.shape)
display(data.head())
display(data.describe().T)


In [ ]:
print("Missing values:")
display(data.isna().sum().to_frame("missing"))
print("Duplicate rows:", data.duplicated().sum())


## 2. Target distribution and unit conversion

`SalePrice` is stored in units of $100,000. A value of `4.25` represents
approximately `$425,000`. The source target is capped near `$500,001`.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(data["SalePrice"] * TARGET_MULTIPLIER, bins=40)
plt.xlabel("Median house value (USD)")
plt.ylabel("Frequency")
plt.title("California Housing Target Distribution")
plt.show()


## 3. Outlier diagnostics

The project reports IQR outliers but retains observations by default. Extreme
values may represent real geographic market segments, and blind deletion can
damage generalization.


In [ ]:
outlier_report = build_iqr_outlier_report(data)
display(outlier_report)


## 4. Optional exploratory features

The deployed ANN uses the original eight predictors to remain compatible with
the supplied model. These ratios are exploratory candidates for future retraining.


In [ ]:
exploratory = add_optional_exploratory_features(data[FEATURE_COLUMNS])
display(exploratory[["RoomsPerBedroom", "IncomePerOccupant", "EstimatedHouseholds"]].describe().T)


## 5. Recreate the deterministic held-out test split

In [ ]:
x_train, x_validation, x_test, y_train, y_validation, y_test = split_housing_data(data)
print("Train:", x_train.shape)
print("Validation:", x_validation.shape)
print("Test:", x_test.shape)


## 6. Load the saved ANN pipeline and evaluate it

In [ ]:
pipeline = PredictionPipeline()
test_predictions = pipeline.predict_target_units(x_test)
metrics = regression_metrics(y_test.to_numpy(), test_predictions)
display(pd.Series(metrics, name="value").to_frame())


In [ ]:
comparison = x_test.reset_index(drop=True).copy()
comparison["ActualPriceUSD"] = y_test.reset_index(drop=True) * TARGET_MULTIPLIER
comparison["PredictedPriceUSD"] = test_predictions * TARGET_MULTIPLIER
comparison["AbsoluteErrorUSD"] = (
    comparison["ActualPriceUSD"] - comparison["PredictedPriceUSD"]
).abs()
display(comparison.head(10))


## 7. Portfolio evaluation visuals

In [ ]:
from IPython.display import Image, display

for filename in [
    "actual_vs_predicted.png",
    "residual_plot.png",
    "feature_importance.png",
    "price_distribution.png",
]:
    path = OUTPUT_DIR / filename
    print(filename)
    display(Image(filename=str(path)))


## 8. Business-friendly single prediction

In [ ]:
sample_values = x_test.iloc[0].to_dict()
result = pipeline.predict_one(sample_values)

print(f"Predicted value: ${result['prediction_usd']:,.0f}")
print(
    f"Empirical range: ${result['estimated_low_usd']:,.0f} - "
    f"${result['estimated_high_usd']:,.0f}"
)
print("Category:", result["category"])
display(pd.DataFrame(result["local_drivers"]))
print(result["interpretation"])


## 9. Batch prediction

The same saved pipeline supports CSV scoring, price categories, empirical ranges,
and downloadable output in the Streamlit application.


In [ ]:
batch_output = pipeline.predict_batch(x_test.head(10))
display(batch_output)


## 10. Retraining

Run the modular training script from the project root:

```bash
python -m src.model_training --data-path data/house_prices.csv
```

The script tunes the ANN, applies early stopping, saves artifacts, generates
evaluation visuals, and refreshes metadata. TensorFlow and hardware differences
can cause small numerical variation.
